# Backend walkthrough — the new portal layer

This notebook imports `backend/main.py` **as a library** (no server starts)
and exercises the new endpoints function-by-function, so you can read the
code *and* see real outputs side by side.

| What | Where in `main.py` | Tried below |
|---|---|---|
| CORS middleware (lets Vite `:5173` call the API) | app setup | — |
| `keyed()` — re-links schedules via canonical keys | portal helpers | Cell 3 |
| `portal_loan_type / portal_total_due / split_name / months_since` | portal helpers | Cell 3 |
| `GET /portal/members` | portal endpoints | Cell 5 |
| `GET /portal/loans` | portal endpoints | Cell 7 |
| `GET /portal/dashboard` | portal endpoints | Cell 9 |
| `GET /risk/delinquent`, `/analytics/by-term`, `/reports/summary` | analytics endpoints | Cell 11 |

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/home/gustave/Documents/program_systems/ucms_frontend/capstone-main")
sys.path.insert(0, str(ROOT / "backend"))
sys.path.insert(0, str(ROOT / "scripts"))

import main as api  # backend/main.py — import only, uvicorn does NOT start

paths = sorted({r.path for r in api.app.routes if hasattr(r, "path")})
print(f"{len(paths)} routes registered:")
print("\n".join(paths))

## Portal helpers — the translation layer

- `keyed()` recomputes canonical loan keys so schedule rows actually link to loans
  (without it, paid sums stay 0 and everything looks delinquent).
- `portal_loan_type()` maps CSV `"Emergency Loan"/"N/A"` to the frontend's six
  types, flagging guesses with `type_source="inferred"`.
- `portal_total_due()` is the same diminishing closed form as
  `web/src/utils/amortization.js`.
- `months_since()` mirrors `loans.js monthsSince()` (whole calendar months).

In [ ]:
import inspect

for name in ["keyed", "portal_loan_type", "portal_total_due",
             "split_name", "months_since"]:
    print("=" * 70)
    print(inspect.getsource(getattr(api, name)))

## `GET /portal/members` — MembersView-shaped rows

Names split, `KHS-0001…` numbering, unknown CSV fields stay `""` / `0` / `None`.

In [ ]:
import pandas as pd

members = api.portal_members()
mdf = pd.DataFrame(members)
print(len(mdf), "members")
mdf[["memberNo", "memberName", "firstName", "lastName",
     "status", "dateJoined", "n_loans", "totalBorrowed"]].head(10)

## `GET /portal/loans` — LoansView-shaped rows

Full loan objects: `loanNo`, `type`, `principal`, `ratePercent`, `termMonths`,
`totalDue`, `paidTotal`, `balance`, `status` (Paid/Released), `delinquent`,
`payments[]`, `history[]`. Try the filters: `api.portal_loans(status="Paid")`.

In [ ]:
import json

loans = api.portal_loans()
print(len(loans), "loans")
first_open = [l for l in loans if l["status"] == "Released"][0]
print(json.dumps(first_open, indent=1, ensure_ascii=False)[:1400])

## `GET /portal/dashboard` — the four cards + attention + activity

Same rules as the Vue computed props. `dailyIncome`/`monthIncome` are `null`
by design — ledger data lives in Supabase, and the API returns `null` with
an `income_note` instead of inventing numbers.

In [ ]:
dash = api.portal_dashboard()
print(json.dumps(dash["cards"], indent=1))
print("attention:", dash["attention"])
print("--- recent activity ---")
for e in dash["recentActivity"][:5]:
    print(e["at"], "|", e["label"], "|", e["action"][:65])

## Risk + charts — analytics endpoints with live plots

`monthsIdle` here uses the same 6-month rule as `loans.js isDelinquent()`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

bt = api.by_term()
labels = [str(int(r["term_months"])) for r in bt]
plt.figure(figsize=(13, 4))
plt.bar(labels, [r["total"] / 1000 for r in bt], color="#2E8B57")
plt.title("Total disbursed per term (thousands) — live from GET /analytics/by-term")
plt.xlabel("Term (months)")
plt.ylabel("Thousand pesos")
plt.tight_layout()
plt.show()

In [ ]:
dl = api.delinquent_list(limit=5)
for d in dl:
    print(d["band"], d["score"], "|", d["member"], "|", "; ".join(d["reasons"][:2]))
print()
s = api.reports_summary()
print(json.dumps({k: v for k, v in s.items() if k != "by_type"}, indent=1))